# 구매이력 기반 N/V 차원별 조건화 M2 — Dunnhumby seed 42

기존 양성 후보에서 두 개의 rank-4 변환을 제거하고, N/V별 64차원 곱셈 벡터만 학습합니다.

- 학습: Dunnhumby 1~683일
- 평가: 684~690일의 신규 상품
- 구매이력: 같은 이진 그래프에서 학습되는 아이템 ID 임베딩의 1-hop 집계 `H_u`
- 조건: `c_N=q_N-mean(q_N)`, `c_V=q_V-mean(q_V)` (train 유효 사용자 기준)
- 보정: `D_u=H_u ⊙ [c_N(u)w_N+c_V(u)w_V]`
- 사용자 표현: `NormPreserve(E_u + 0.05 D_u)`
- 추가 파라미터: `w_N,w_V` 합계 128개
- 고정: binary graph, uniform negative sampling, plain BPR, 100 epoch, 하나의 optimizer
- 비교: 동일 protocol의 기존 M1@64 결과 재사용

이 실행은 역사적 개발구간 seed 42 탐색이며 통계적 유의성이나 test 일반화를 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'e1009e326b9c8f72cd6af39ae5723444dc21b356'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)

In [ ]:
import json
import torch
from lightgcn_clv_neighbor_conditioned_featurewise import (
    configure_neighbor_conditioned_featurewise_run,
    preflight_summary,
    run_neighbor_conditioned_featurewise_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_neighbor_conditioned_featurewise_run(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_neighbor_conditioned_featurewise_historical_screen_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['m2']['embedding_dim'] == 64
assert summary['m2']['axis_parameter_count'] == 128
assert summary['m2']['rho'] == 0.05
assert summary['m2']['condition_centring'] == 'valid_train_user_mean'
assert summary['m2']['explicit_item_features'] is False
assert summary['m2']['item_transformation'] is False
assert summary['fixed']['graph'] == 'binary'
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['one_training_loop_and_optimizer'] is True
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_neighbor_conditioned_featurewise_screen(cfg)

In [ ]:
from IPython.display import display

comparison = result_df.attrs['comparison'].copy()
reading = dict(result_df.attrs['screening_reading'])
paths = dict(result_df.attrs['result_paths'])
display_df = result_df.copy()
display_df.attrs = {}

print('절대지표:')
display(display_df.sort_values('model_id'))

core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('M1@64 대비 핵심 변화:')
display(comparison[comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('탐색 판독:', reading)
print('결과 파일:', paths)